# Qualitätsprüfung: IPA-Modell (neurlang/ipa-whisper-base)

Dieses Notebook prüft, ob die IPA-Transkriptionen (Modell B) brauchbar sind.

## Stufe 1: Schnelle Sichtprüfung

10–20 zufällige Samples anschauen: Enthält der Output IPA-Zeichen?

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

df = pd.read_csv('Data/transcriptions.csv')
print(f'Geladen: {len(df):,} Zeilen')
print(f'Spalten: {list(df.columns)}')
print()

Geladen: 1,400 Zeilen
Spalten: ['path', 'dialect_region', 'sentence', 'transcription_whisper', 'transcription_ipa']



In [2]:
# 3 zufällige Samples pro Dialektregion
samples = df.groupby('dialect_region', group_keys=False).apply(
    lambda x: x.sample(3, random_state=42)
)

for _, row in samples.iterrows():
    print(f"🗺  {row['dialect_region']}")
    print(f"   HD:      {row['sentence']}")
    print(f"   Whisper: {row['transcription_whisper']}")
    print(f"   IPA:     {row['transcription_ipa']}")
    print()

KeyError: 'dialect_region'

### Was du suchst:
- Enthält der IPA-Output spezielle Zeichen? (`ə`, `ʊ`, `ɪ`, `ː`, `ˈ` etc.)
- Oder gibt das Modell normalen Text / Hochdeutsch zurück?
- Gibt es viele leere oder sehr kurze Outputs?

In [3]:
# Schnellcheck: Leere / kurze Outputs
df['ipa_len'] = df['transcription_ipa'].fillna('').str.len()
df['whisper_len'] = df['transcription_whisper'].fillna('').str.len()

empty_ipa = (df['ipa_len'] == 0).sum()
short_ipa = (df['ipa_len'] < 5).sum()

print(f'Leere IPA-Outputs:    {empty_ipa} / {len(df)}')
print(f'Sehr kurze (<5 Zeichen): {short_ipa} / {len(df)}')
print(f'Durchschnittliche Länge IPA:     {df["ipa_len"].mean():.0f} Zeichen')
print(f'Durchschnittliche Länge Whisper:  {df["whisper_len"].mean():.0f} Zeichen')

Leere IPA-Outputs:    0 / 1400
Sehr kurze (<5 Zeichen): 0 / 1400
Durchschnittliche Länge IPA:     54 Zeichen
Durchschnittliche Länge Whisper:  52 Zeichen


## Stufe 2: Quantitative Checks – IPA-Anteil messen

In [4]:
# IPA-spezifische Zeichen (nicht in normalem Deutsch-Text)
IPA_SPECIAL = set('əɛɪʊɔœæɑɐʌɜɒðŋɡʃʒθβχɣɸɹɻɺɾʀʁʋʍɥɰɫɬɮɭɳɵɤɯʔːˈˌ')

def ipa_ratio(text):
    """Anteil IPA-spezifischer Zeichen (die NICHT in normalem Text vorkommen)."""
    if not isinstance(text, str) or len(text) == 0:
        return 0
    ipa_count = sum(1 for c in text if c in IPA_SPECIAL)
    return ipa_count / len(text)

df['ipa_ratio'] = df['transcription_ipa'].apply(ipa_ratio)
df['whisper_ipa_ratio'] = df['transcription_whisper'].apply(ipa_ratio)

print('=== IPA-Anteil pro Region (Modell B: IPA) ===')
print(df.groupby('dialect_region')['ipa_ratio'].describe().round(3))
print()
print('=== IPA-Anteil pro Region (Modell A: Whisper – sollte ~0 sein) ===')
print(df.groupby('dialect_region')['whisper_ipa_ratio'].describe().round(3))

=== IPA-Anteil pro Region (Modell B: IPA) ===
                count   mean    std    min    25%    50%    75%    max
dialect_region                                                        
Basel           200.0  0.440  0.068  0.095  0.405  0.447  0.480  0.615
Bern            200.0  0.432  0.077  0.100  0.404  0.444  0.477  0.604
Graubünden      200.0  0.421  0.075  0.000  0.395  0.432  0.464  0.583
Innerschweiz    200.0  0.423  0.053  0.188  0.395  0.422  0.453  0.600
Ostschweiz      200.0  0.435  0.083  0.000  0.410  0.440  0.479  0.652
Wallis          200.0  0.411  0.082  0.020  0.380  0.431  0.464  0.595
Zürich          200.0  0.424  0.066  0.089  0.399  0.435  0.462  0.583

=== IPA-Anteil pro Region (Modell A: Whisper – sollte ~0 sein) ===
                count  mean  std  min  25%  50%  75%  max
dialect_region                                           
Basel           200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
Bern            200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
Graubünden     

In [ ]:
# Visualisierung: IPA-Anteil
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramm IPA-Ratio
axes[0].hist(df['ipa_ratio'], bins=30, color='#DD8452', edgecolor='white', alpha=0.8)
axes[0].axvline(0.1, color='red', linestyle='--', label='Schwelle 10%')
axes[0].set_xlabel('IPA-Zeichenanteil')
axes[0].set_ylabel('Anzahl Samples')
axes[0].set_title('Verteilung des IPA-Anteils (Modell B)')
axes[0].legend()

# Boxplot pro Region
regions = df['dialect_region'].unique()
data = [df[df['dialect_region'] == r]['ipa_ratio'].values for r in sorted(regions)]
bp = axes[1].boxplot(data, labels=sorted(regions), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#DD8452')
    patch.set_alpha(0.6)
axes[1].set_ylabel('IPA-Zeichenanteil')
axes[1].set_title('IPA-Anteil pro Dialektregion')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Erwartung: IPA-Anteil > 0.10 = Modell gibt echte IPA-Zeichen aus')
print('           IPA-Anteil ~ 0    = Modell gibt normalen Text aus')

## Stufe 3: Bekannte Schweizerdeutsche Laute prüfen

Gezielt nach Dialektmerkmalen suchen, die im IPA auftauchen sollten.

In [5]:
ipa_col = df['transcription_ipa'].fillna('')

checks = {
    'hat /x/ (ch-Laut, z.B. "ach")':     ipa_col.str.contains('x'),
    'hat lange Vokale /ː/':               ipa_col.str.contains('ː'),
    'hat /ʃ/ (sch-Laut)':                 ipa_col.str.contains('ʃ'),
    'hat /r/-Varianten (r/ɾ/ʀ/ʁ)':       ipa_col.str.contains('[rɾʀʁ]', regex=True),
    'hat Schwa /ə/':                      ipa_col.str.contains('ə'),
    'hat /ɪ/ (kurzes i)':                 ipa_col.str.contains('ɪ'),
    'hat /ʊ/ (kurzes u)':                 ipa_col.str.contains('ʊ'),
    'hat Betonungszeichen /ˈ/':           ipa_col.str.contains('ˈ'),
}

print('=== Häufigkeit bekannter IPA-Laute (über alle Samples) ===')
for label, mask in checks.items():
    print(f'  {label}: {mask.mean():.1%}')

# Pro Region aufschlüsseln
print()
print('=== /x/ (ch-Laut) pro Region ===')
print(df.groupby('dialect_region').apply(
    lambda g: g['transcription_ipa'].fillna('').str.contains('x').mean()
).round(3).to_string())

print()
print('=== /ʃ/ (sch-Laut) pro Region ===')
print(df.groupby('dialect_region').apply(
    lambda g: g['transcription_ipa'].fillna('').str.contains('ʃ').mean()
).round(3).to_string())

=== Häufigkeit bekannter IPA-Laute (über alle Samples) ===
  hat /x/ (ch-Laut, z.B. "ach"): 28.3%
  hat lange Vokale /ː/: 91.1%
  hat /ʃ/ (sch-Laut): 53.6%
  hat /r/-Varianten (r/ɾ/ʀ/ʁ): 90.1%
  hat Schwa /ə/: 66.3%
  hat /ɪ/ (kurzes i): 80.9%
  hat /ʊ/ (kurzes u): 48.1%
  hat Betonungszeichen /ˈ/: 90.3%

=== /x/ (ch-Laut) pro Region ===
dialect_region
Basel           0.310
Bern            0.295
Graubünden      0.195
Innerschweiz    0.295
Ostschweiz      0.290
Wallis          0.330
Zürich          0.265

=== /ʃ/ (sch-Laut) pro Region ===
dialect_region
Basel           0.570
Bern            0.495
Graubünden      0.585
Innerschweiz    0.465
Ostschweiz      0.530
Wallis          0.595
Zürich          0.515


In [ ]:
# Visualisierung: Häufigkeit der IPA-Laute pro Region
laut_checks = {
    '/x/': 'x',
    '/ː/': 'ː',
    '/ʃ/': 'ʃ',
    '/ə/': 'ə',
    '/ɪ/': 'ɪ',
    '/ʊ/': 'ʊ',
}

regions_sorted = sorted(df['dialect_region'].unique())
results = {}
for label, char in laut_checks.items():
    results[label] = [
        df[df['dialect_region'] == r]['transcription_ipa'].fillna('').str.contains(
            char.replace('/', ''), regex=False
        ).mean()
        for r in regions_sorted
    ]

result_df = pd.DataFrame(results, index=regions_sorted)

fig, ax = plt.subplots(figsize=(12, 6))
result_df.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_ylabel('Anteil Samples mit diesem Laut')
ax.set_title('Häufigkeit ausgewählter IPA-Laute pro Dialektregion')
ax.set_xlabel('Dialektregion')
ax.legend(title='IPA-Laut', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Stufe 4: Vergleich Whisper vs. IPA als Sanity-Check

Wenn Whisper "nicht" transkribiert, sollte IPA etwas wie `/nɪd/`, `/nøːd/` oder `/nɛt/` zeigen – **nicht** `/nɪçt/` (das wäre Hochdeutsch-Phonologie).

In [ ]:
# Samples wo Whisper "nicht" transkribiert hat
nicht_mask = df['transcription_whisper'].fillna('').str.contains('nicht', case=False)
nicht_samples = df[nicht_mask]

print(f'Samples mit "nicht" in Whisper-Output: {len(nicht_samples)}')
print()

# Pro Region ein paar Beispiele
for region in sorted(nicht_samples['dialect_region'].unique()):
    region_samples = nicht_samples[nicht_samples['dialect_region'] == region].head(2)
    for _, row in region_samples.iterrows():
        print(f'🗺  {row["dialect_region"]}')
        print(f'   Whisper: {row["transcription_whisper"]}')
        print(f'   IPA:     {row["transcription_ipa"]}')
        print()

In [ ]:
# Weitere Sanity-Checks: Bekannte Wörter vergleichen
test_words = ['und', 'der', 'ist', 'auch', 'das']

for word in test_words:
    mask = df['transcription_whisper'].fillna('').str.contains(f'\\b{word}\\b', case=False, regex=True)
    count = mask.sum()
    if count > 0:
        example = df[mask].iloc[0]
        print(f'Whisper "{word}" ({count}x) → IPA: {example["transcription_ipa"][:80]}...')
        print()

## Zusammenfassung

Checke folgende Punkte:

| Check | Erwartet | Schlecht |
|-------|----------|--------|
| IPA-Zeichenanteil | > 10% | ~ 0% (normaler Text) |
| Leere Outputs | < 5% | > 20% |
| Bekannte Laute (/x/, /ː/, /ʃ/) | > 30% der Samples | < 5% |
| "nicht" → IPA | /nɪd/, /nøːd/ | /nɪçt/ (= Hochdeutsch) |
| Unterschiede zwischen Regionen | Sichtbar | Alle gleich |